# Zarr File Reader Notebook

This notebook demonstrates how to import and use dependencies for reading zarr files.

## Import Required Libraries

Import the necessary libraries for working with zarr files, including zarr, numpy, and pandas.

In [1]:
import zarr
import numpy as np
import pandas as pd
import dask.dataframe as dd
import pyarrow as pa
from pathlib import Path
import sys
import os

print(sys.executable)
print(f"Python version: {sys.version}")
print(f"Zarr version: {zarr.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("PyArrow version:", pa.__version__)
print("PyArrow location:", os.path.dirname(pa.__file__))
print("\nAll dependencies installed successfully!")

c:\Users\jrand\projects\TranscriptCounting\.venv\Scripts\python.exe
Python version: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Zarr version: 2.18.3
NumPy version: 2.2.6
Pandas version: 2.3.3
PyArrow version: 20.0.0
PyArrow location: c:\Users\jrand\projects\TranscriptCounting\.venv\lib\site-packages\pyarrow

All dependencies installed successfully!


## Setup file paths
Set the name of the folder path we should be using to find the associated data files

In [2]:
topLevelFolder = "20250827__205437__CSU_Bouchet_v1_2025-08-27"
subfolder = "output-XETG00230__0063690__Region-2E__20250827__205536"

transcriptsZarrPath = Path(topLevelFolder) / Path(subfolder) / "transcripts.zarr.zip"
transcriptsParquetPath =  Path(topLevelFolder) / Path(subfolder) / "transcripts.parquet"

## Get Transcript Counts

Note: 10x refers to the transcripts as "feature_name"

1. Open the region summary csv and get the list of cell IDs we're interested in. Ignore any data that starts with a "#"
2. Create a results csv file
3. Read the transcripts parquet file, and read only the columns that are important for the analysis
4. Filter the data by removing any data where the quality score is less than 20 (matches 10x behavior) and remove any "unassigned" or "negcontrol" feature names
5. Create a pivot table of the data based on cell_id and transcript
6. Output the pivot table to the csv document
7. Print out any cell IDs that were found in the region summary, but did not appear in the parquet file.

In [3]:
# read the first column of the Reion-2e file into an array
# ignore rows that start with #
regionSummary = pd.read_csv("Region-2E L PH_cells 2025Dec05.csv", comment='#')

# create a new csv for the results
results_csv_path = "transcript_counts_results.csv"

#delete the file if it already exists
if os.path.exists(results_csv_path):
    os.remove(results_csv_path)

# read parquet file with dask
fileData = dd.read_parquet(transcriptsParquetPath, index=False, engine='pyarrow', columns=['cell_id', 'feature_name', 'qv'])

# only keep data with a q score of 20 or higher to match 10x defaults
fileData = fileData[fileData["qv"] >= 20]

# filter the data to only include valid transcripts
def filterValidTranscripts(cell_data):
    #filter out feature names that contain "unknown"
    cell_data = cell_data[~cell_data["feature_name"].str.contains("UnassignedCodeword_")]

    #filter out feature names that contain "NegControl"
    cell_data = cell_data[~cell_data["feature_name"].str.contains("NegControl")]
    return cell_data

# get the list of transcripts 
fileData_filtered = filterValidTranscripts(fileData)
cell_gene_matrix = (
    fileData_filtered
    .groupby(["cell_id", "feature_name"])
    .size()
    .compute()
    .unstack(fill_value=0)
)

valid_cell_ids = regionSummary['Cell ID'][regionSummary['Cell ID'].isin(cell_gene_matrix.index)]
results_df = cell_gene_matrix.loc[valid_cell_ids].sort_index()
results_df.to_csv(results_csv_path)

# Print which cells were not found 
missing_cells = regionSummary['Cell ID'][~regionSummary['Cell ID'].isin(cell_gene_matrix.index)]
if len(missing_cells) > 0:
    print(f"Warning: {len(missing_cells)} cells not found in parquet data:")
    print(missing_cells.tolist())


['obffagjj-1', 'nhlbofdi-1', 'nhldjmeh-1', 'gpgadiip-1', 'gpgdpjjj-1', 'gpnglngf-1', 'nibbjjoi-1', 'nibcdlfi-1', 'nhpgnhdh-1', 'gpjibclp-1', 'gpjdfjjm-1', 'gpjckgii-1', 'gpggklac-1']
